# ARCANA front matter loss — interactive reproduction

Companion notebook to [`arcana_frontmatter_repro.py`](arcana_frontmatter_repro.py)
(**keep both files in the same folder** — the notebook imports its helpers, so
the two stay in sync). It walks the same steps one by one and shows the **full
raw JSON** of every response, so the evidence can be inspected directly.

**The problem.** Markdown files uploaded to an ARCANA may carry a YAML metadata
header ("front matter") — the same mechanism GWDG's own Docling pipeline uses for
the ["Markdown Plus" metadata header](https://docs.hpc.gwdg.de/services/ai-services/arcana/docling-process/)
(`Author`, `Title`, `Description`, …). That metadata does **not** survive
retrieval: the `References:` block of an ARCANA-routed chat reply carries only
`[RREFn] <filename> (<distance>)` plus the chunk body starting at its first
heading, and no response field — structured or text — returns the per-chunk
metadata. The filename is the only metadata channel that survives.

**The test.** Upload one markdown file whose metadata header mirrors the
documented Docling example field for field, with sentinel values in the three
free-text fields that occur nowhere in the body. Then ask a question only that
file can answer and search the entire raw response JSON for the sentinels —
once via the OpenAI SDK, once via a plain HTTP POST (rules out an SDK artifact).

**Requirements:** `pip install requests openai` (`openai` optional) and a SAIA
API key in the `SAIA_API_KEY` environment variable.

In [1]:
import json
import os

from arcana_frontmatter_repro import (
    BODY_PHRASE,
    DEFAULT_BASE_URL,
    DEFAULT_MODEL,
    FILE_NAME,
    QUESTION,
    SENTINEL_VALUES,
    available_models,
    build_document,
    chat_via_openai_sdk,
    chat_via_plain_http,
    create_arcana,
    delete_arcana,
    download_stored_file,
    generate_index_and_wait,
    message_content,
    sentinels_found,
    upload_document,
)

BASE_URL = DEFAULT_BASE_URL
MODEL = DEFAULT_MODEL

API_KEY = os.environ.get("SAIA_API_KEY", "")
if not API_KEY:
    # Convenience fallback when saia_python is installed (NOT required):
    # resolve the key like the library does, also trying the repo-root
    # .env since this notebook's kernel usually starts in examples/.
    try:
        from saia_python.auth import load_api_key
    except ImportError:
        load_api_key = None
    if load_api_key is not None:
        for candidate in (None, "../.env"):
            try:
                API_KEY = load_api_key(candidate)
                break
            except Exception:
                continue
if not API_KEY:
    raise RuntimeError("set SAIA_API_KEY before running this notebook")

# An unknown model makes the gateway answer with an opaque 500 — check first.
models = available_models(BASE_URL, API_KEY)
if models is not None and MODEL not in models:
    raise RuntimeError(f"model {MODEL!r} not available — pick one of: {models}")
print("base URL:", BASE_URL)
print("model:   ", MODEL)

base URL: https://chat-ai.academiccloud.de/v1
model:    openai-gpt-oss-120b


## 1 — The test document

The metadata header mirrors the documented Docling "Markdown Plus" example
(`Author`, `Title`, `Description`, `Filename`, `Extension`, `Number of Pages`,
`Version`). The `FM-SENTINEL-…` values appear **only** in the header, never in
the body.

In [2]:
document = build_document()
print(document)

---
Author: FM-SENTINEL-7D3F-AUTHOR
Title: FM-SENTINEL-7D3F-TITLE
Description: FM-SENTINEL-7D3F-DESCRIPTION
Filename: frontmatter_repro_doc.md
Extension: md
Number of Pages: 1
Version: 1.0
---

# Maintenance of the Quendelburg lighthouse

The fictional Quendelburg lighthouse is repainted every seven years with a
weather-resistant mixture of linseed oil and crushed mussel shells, applied
in three layers by the keepers' guild.

The lantern room is cleaned every spring; the Fresnel lens is inspected by
two independent opticians every other year.



## 2 — Create a throwaway arcana, upload, index

Management API (`/arcanas/api/v1`, raw-key auth). Indexing of the single small
file usually finishes within a few minutes.

In [3]:
arcana_name, arcana_id = create_arcana(BASE_URL, API_KEY)
print("created:", arcana_id)
upload_document(BASE_URL, API_KEY, arcana_name, document)
print("uploaded:", FILE_NAME)
generate_index_and_wait(BASE_URL, API_KEY, arcana_name)

created: schwarz89/fm-repro-3cc1afa2
uploaded: frontmatter_repro_doc.md
    index status: INDEXED


## 3 — Control: the stored file still contains the front matter

The management API's `…/files/<name>/download` endpoint returns the stored file
**with** the metadata header — so it is not stripped at upload; the loss happens
at index/retrieval time.

In [4]:
stored = download_stored_file(BASE_URL, API_KEY, arcana_name)
print(stored)
stored_has_fm = all(v in stored for v in SENTINEL_VALUES)
print("front matter present in stored copy:", stored_has_fm)

---
Author: FM-SENTINEL-7D3F-AUTHOR
Title: FM-SENTINEL-7D3F-TITLE
Description: FM-SENTINEL-7D3F-DESCRIPTION
Filename: frontmatter_repro_doc.md
Extension: md
Number of Pages: 1
Version: 1.0
---

# Maintenance of the Quendelburg lighthouse

The fictional Quendelburg lighthouse is repainted every seven years with a
weather-resistant mixture of linseed oil and crushed mussel shells, applied
in three layers by the keepers' guild.

The lantern room is cleaned every spring; the Fresnel lens is inspected by
two independent opticians every other year.

front matter present in stored copy: True


## 4 — Chat via the OpenAI SDK — full raw response JSON

The ARCANA-specific request pieces: `extra_body={"arcana": {"id": …},
"enable-tools": True}` plus the `inference-service: saia-openai-gateway` header
(without the header the request returns 200 OK but retrieval silently never
fires). In the JSON below, note that `choices[0].message.content` carries the
answer **and** the appended `References:` block — the retrieved chunk body is
included verbatim, starting at its first `#` heading, with the metadata header
gone. No other field carries it.

In [5]:
print("question:", QUESTION)
sdk_resp = chat_via_openai_sdk(BASE_URL, API_KEY, MODEL, arcana_id)
if sdk_resp is None:
    print("`openai` not installed — this leg skipped (plain HTTP below suffices)")
else:
    print(json.dumps(sdk_resp, indent=2, ensure_ascii=False))

question: According to the documents, how often is the Quendelburg lighthouse repainted, and what paint mixture is used?
{
  "id": "chatcmpl-a439358eed049ee8",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "According to the document, the Quendelburg lighthouse is **repainted every seven years**. The keepers’ guild applies a **weather‑resistant paint mixture of linseed oil and crushed mussel shells** (in three layers) for each repainting cycle【RREF1】.\n-----------------------------------------\nReferences:\n\n[RREF1] frontmatter_repro_doc.md (0.135)\n# Maintenance of the Quendelburg lighthouse\n\nThe fictional Quendelburg lighthouse is repainted every seven years with a\nweather-resistant mixture of linseed oil and crushed mussel shells, applied\nin three layers by the keepers' guild.\n\nThe lantern room is cleaned every spring; the Fresnel lens is inspected by\ntwo independent opticians every other ye

## 5 — Same chat via plain HTTP POST (no SDK) — full raw response JSON

Byte-identical request via `requests.post` — demonstrating the loss is not an
OpenAI-SDK artifact but happens server-side.

In [6]:
http_resp = chat_via_plain_http(BASE_URL, API_KEY, MODEL, arcana_id)
print(json.dumps(http_resp, indent=2, ensure_ascii=False))

{
  "id": "chatcmpl-9e08d4a8ffd4f945",
  "object": "chat.completion",
  "created": 1781120691,
  "model": "openai-gpt-oss-120b",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "According to the document, the Quendelburg lighthouse is **repainted every seven years**. The keepers’ guild applies a **weather‑resistant paint mixture of linseed oil and crushed mussel shells** (in three layers) for each repainting cycle【RREF1】.\n-----------------------------------------\nReferences:\n\n[RREF1] frontmatter_repro_doc.md (0.135)\n# Maintenance of the Quendelburg lighthouse\n\nThe fictional Quendelburg lighthouse is repainted every seven years with a\nweather-resistant mixture of linseed oil and crushed mussel shells, applied\nin three layers by the keepers' guild.\n\nThe lantern room is cleaned every spring; the Fresnel lens is inspected by\ntwo independent opticians every other year.\n",
        "refusal": null,
        "annotations": n

## 6 — Verdict

Search the **entire** raw response JSON of both legs for the three sentinel
values. Expected (desired): they are retrievable alongside the `[RREFn]`
reference. Observed: `NONE` — while the chunk *body* text from the very same
file is returned verbatim.

In [7]:
content = message_content(http_resp)
print("retrieval fired (file cited):       ", "[RREF" in content and FILE_NAME in content)
print("chunk BODY returned verbatim:       ", BODY_PHRASE in content)
print("stored file keeps front matter:     ", stored_has_fm)
for leg, payload in (("openai_sdk", sdk_resp), ("plain_http", http_resp)):
    if payload is None:
        continue
    hits = sentinels_found(payload)
    print(f"front matter sentinels in {leg}:  {hits if hits else 'NONE  <-- the bug'}")

retrieval fired (file cited):        True
chunk BODY returned verbatim:        True
stored file keeps front matter:      True
front matter sentinels in openai_sdk:  NONE  <-- the bug
front matter sentinels in plain_http:  NONE  <-- the bug


## 7 — Cleanup

Deletes the throwaway arcana. **Skip this cell** to keep it for server-side
inspection.

In [8]:
delete_arcana(BASE_URL, API_KEY, arcana_name)
print("deleted:", arcana_id)

deleted: schwarz89/fm-repro-3cc1afa2
